# Домашнее задание: Функции

## Задание 1: Конвертер регистров

Написать функцию, которая будет переводить snake_case в PascalCase и наоборот. 

Функция должна сама определять - какой формат ей передали. Можно добавить ключевой аргумент, который будет принудительно возвращать один из форматов.

<br>

**Примеры:**
* `otus_course     -> OtusCourse`
* `PythonIsTheBest -> python_is_the_best`



In [11]:
def convert_case(source: str, case: str = "auto") -> str:
    if case not in ["pascal", "snake"]:
        # Auto detect
        case = "pascal" if source.find("_") != -1 or source.islower() else "snake"

    words: list[str] = []
    last = 0
    for index, ch in enumerate(source):
        word = None
        if ch == "_":
            word = source[last:index]
            last = index + 1
        elif ch.isupper():
            word = source[last:index]
            last = index

        if word:
            word = word.capitalize() if case == "pascal" else word.lower()
            words.append(word)

    word = source[last:]
    word = word.capitalize() if case == "pascal" else word.lower()
    words.append(word)

    if case == "pascal":
        return "".join(words)

    return "_".join(words)

value = input("Test: ")
converted = convert_case(value)
print(f"{value} -> {converted}")

hello_world -> HelloWorld


## Задание 2: Проверка валидности даты

Написать функцию проверяющую валидность введенной даты.

<br>

**Примеры:**
* `29.02.2000 -> True`
* `29.02.2001 -> False`
* `31.04.1962 -> False`



In [ ]:
MONTH_DAYS = {
    1: 31,  # Jan
    2: 28,  # Feb
    3: 31,  # Mar
    4: 30,  # Apr
    5: 31,  # May
    6: 30,  # Jun
    7: 31,  # Jul
    8: 31,  # Aug
    9: 30,  # Sep
    10: 31, # Oct
    11: 30, # Nov
    12: 31, # Dec
}

def is_date(date: str) -> bool:
    day = 0
    month = 0
    year = 0

    part = 0
    last = 0
    for index, ch in enumerate(date):
        if ch == ".":
            if index == last or index - last > 2:
                return False

            match part:
                case 0:
                    day = int(date[last:index])
                case 1:
                    month = int(date[last:index])
                case _:
                    return False
            part += 1
            last = index + 1
        elif ch < "0" or "9" < ch:
            return False

    year = int(date[last:])

    # В календаре нет нулевого года вместо него 1 год до н.э.
    if year <= 0:
        return False
    if month not in  MONTH_DAYS:
        return False

    is_leap = year % 400 == 0 or (year % 100 != 0 and year % 4 == 0)
    days = MONTH_DAYS[month]
    if month == 2 and is_leap:
        days += 1

    return 1 <= day <= days

value = input("Test:")
print(f"Input: {value}\nIs date: {is_date(value)}")


Input: 31.04.1962
Is date: False


## Задание 3: Проверка на простое число

Функция проверки на простое число. Простые числа – это такие числа, которые делятся на себя и на единицу.

<br>

**Примеры:**
* `17 -> True`
* `20 -> False`
* `23 -> True`

In [ ]:
from math import isqrt

def is_prime(value: int) -> bool:
    """
    Существует два основных типа проверок на простоту:
    1. Основанные на просеивании: Например Решето Эратосфена или Аткина.
       Они медленные но дают гарантированный результат
    2. Основанные на свойствах чисел. Этот класс позволяет:
       1. Либо установить вероятность того, что число является простым (Тест Солвея - Штрассена).
       2. Гарантированно проверять простоту, но только определенного класса чисел (Тест Люка - Лемера)

    Для простоты выбран вариант с исключёнными чётными и проверкой деления на простые
    """
    if value <= 1:
        return False
    if value == 2:
        return True
    if value % 2 == 0:
        return False

    upper_bound = isqrt(value)
    sieve = [True] * (upper_bound // 2 + 1)
    for i in range(1, len(sieve)):
        if not sieve[i]:
            continue

        prime = i * 2 + 1
        if value % prime == 0:
            return False

        for j in range(i, len(sieve), prime):
            sieve[j] = False


    return True

num = int(input("Test: "))
if is_prime(num):
    print(f"{num} is prime")
else:
    print(f"{num} is not prime")


961 is not prime


## Задание 4: Учет пользователей

Пользователь в бесконечном цикле вводит данные пользователей: имя, затем фамилию, возраст и ID. Ввод продолжается до тех пор, пока не будет введено пустое поле. 

Пользователи заносятся в словарь, где ключ это ID пользователя, а остальные данные записываются в виде кортежа. 

**Программа должна проверять:**
* имя и фамилия состоят только из символов и начинаются с большой буквы - если не с большой, то заменяет букву на большую;
* возраст должен быть числом от 18 до 60;
* ID - целое число, дополненное до 8 знаков незначащими нулями, ID должен быть уникальным.

**Дополнительно:** написать функцию, которая будет выводить полученный словарь в виде таблицы.

In [9]:
from typing import Callable

UserStore = dict[str, tuple[str, str, int]]

def ensure_name(value: str) -> str:
    if not value.isalpha():
        raise ValueError("Не слово")
    
    return value.capitalize()


def ensure_age(value: str) -> int:
    result = int(value)
    if not (18 <= result <= 60):
        raise ValueError("Возраст должен быть в интервале (18-60)")

    return result


def read_field(field: str, cb: Callable[[str], any]) -> any:
    value = input(f"{field} >").strip()
    if not value:
        raise ValueError("end")

    return cb(value)


def read_user(store: UserStore) -> None:
    first_name: str = read_field("Имя", ensure_name)
    last_name: str = read_field("Фамилия", ensure_name)
    age: int = read_field("Возраст", ensure_age)
    id: str = read_field("ID", lambda val: f"{int(val):08d}")

    if id in store:
        raise ValueError(f"ID {id} уже занят")

    store[id] = (first_name, last_name, int(age))


def read_users():
    reading = True
    result: UserStore = {}
    while reading:
        try:
            read_user(result)
        except ValueError:
            break

    return result


def print_line(data: (list[str] | tuple[str, str, str, int]), width: list[int]) -> None:
    print(f"{data[0].rjust(width[0])} {data[1].ljust(width[1])} {data[2].ljust(width[2])}" +
          f" {str(data[3]).rjust(width[3])}")


def print_users(store: UserStore) -> None:
    headers = ["#", "Имя", "Фамилия", "Возраст"]
    col_width = [len(s) for s in headers]

    for id in store:
        user = store[id]
        col_width[0] = max(col_width[0], len(id))
        for index, field in enumerate(user, start = 1):
            col_width[index] = max(col_width[index], len(str(field)))

    print_line(headers, col_width)
    for id in store:
        user = store[id]
        print_line((id, *user), col_width)


store = read_users()
print_users(store)


       # Имя       Фамилия Возраст
00000001 Иван      Иванов       18
00000002 Карапитян Шаваш        57
00000003 Тарас     Бульба       43
